# PySpark inner join

## Source data

The source comes from jupyter-pyspark/f1-sourcefiles

# Inner the data from the results.csv and races.csv 


# Initalise a spark session

In [1]:
# Initalise a spark session
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Fix JAVA_HOME to your actual Java 21 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell"

# Build Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("DeltaLakeExample") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


26/04/14 23:51:49 WARN Utils: Your hostname, DESKTOP-OQT8U26 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/14 23:51:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/robyip/projects/pyspark-deltalake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/robyip/.ivy2/cache
The jars for the packages stored in: /home/robyip/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-915bade9-266f-4e07-b145-0aac99ece1c4;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 174ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0 

# Load results.csv and races.csv into dataframes

In [2]:
# Load csv files into dataframes
#  Contains headers
# Infers schema

results = spark.read.csv("f1-sourcefiles/results.csv", header=True, inferSchema=True)

races = spark.read.csv("f1-sourcefiles/races.csv", header=True, inferSchema=True)


26/04/14 23:52:03 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


# Show the data type of the dataframes


In [7]:
# results 

results.printSchema()

# races

races.printSchema()

root
 |-- resultId: integer (nullable = true)
 |-- raceId: integer (nullable = true)
 |-- driverId: integer (nullable = true)
 |-- constructorId: integer (nullable = true)
 |-- number: string (nullable = true)
 |-- grid: integer (nullable = true)
 |-- position: string (nullable = true)
 |-- positionText: string (nullable = true)
 |-- positionOrder: integer (nullable = true)
 |-- points: double (nullable = true)
 |-- laps: integer (nullable = true)
 |-- time: string (nullable = true)
 |-- milliseconds: string (nullable = true)
 |-- fastestLap: string (nullable = true)
 |-- rank: string (nullable = true)
 |-- fastestLapTime: string (nullable = true)
 |-- fastestLapSpeed: string (nullable = true)
 |-- statusId: integer (nullable = true)

root
 |-- raceId: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- round: integer (nullable = true)
 |-- circuitId: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- date: date (nullable = true)
 |-- time: string (nulla

# Use Alias in an INNER JOIN

In [4]:
# Inner join on same column name i.e. raceid


# Create and alias for a Dataframe just like tabel alias in SQL

results.results.alias("rslt")

races = races.alias("rcs")

# join on raceid and look at shuffle size to optimise on?


df = results.join(races, on="raceid", how="inner").filter().select("resultid", "raceid", "year", "round")

df.show()

+--------+------+----+-----+
|resultid|raceid|year|round|
+--------+------+----+-----+
|       1|    18|2008|    1|
|       2|    18|2008|    1|
|       3|    18|2008|    1|
|       4|    18|2008|    1|
|       5|    18|2008|    1|
|       6|    18|2008|    1|
|       7|    18|2008|    1|
|       8|    18|2008|    1|
|       9|    18|2008|    1|
|      10|    18|2008|    1|
|      11|    18|2008|    1|
|      12|    18|2008|    1|
|      13|    18|2008|    1|
|      14|    18|2008|    1|
|      15|    18|2008|    1|
|      16|    18|2008|    1|
|      17|    18|2008|    1|
|      18|    18|2008|    1|
|      19|    18|2008|    1|
|      20|    18|2008|    1|
+--------+------+----+-----+
only showing top 20 rows

